# Lab: Real 8-GPU Training Traces with HTA

**Goal:** read a *real* multi-rank distributed-training trace two ways — perceptually in Perfetto, and analytically with Meta's HolisticTraceAnalysis — and reconcile the two. The traces are HTA's demo set: an 8-rank vision-transformer training job, one Kineto JSON per rank.

**What you're walking into (no spoilers beyond this):** this trace is *not* a healthy baseline — it has a diagnosable dominant pathology. Your job is to name it from the timeline before the analytics confirm it.

**Protocol:** every exercise asks you to commit a number *from the timeline* before computing it. Write your guess down. The gap between your read and the computed answer is the thing being trained. A ground-truth answer key is at the bottom — do not scroll there first.

Companion: [torch.profiler & HTA tool page](https://sys-design-primer-cvw.pages.dev/trace-reading/tools/3-torch-profiler/). Setup note: HTA installs cleanly on Python 3.10–3.12 with recent pandas; if `pip install` fights your environment, a fresh venv fixes it.

In [ ]:
%pip install -q HolisticTraceAnalysis pandas
import os, urllib.request
TRACE_DIR = "vision_transformer"
os.makedirs(TRACE_DIR, exist_ok=True)
BASE = "https://raw.githubusercontent.com/facebookresearch/HolisticTraceAnalysis/main/tests/data/vision_transformer"
for r in range(8):
    f = f"rank-{r}.json.gz"
    if not os.path.exists(f"{TRACE_DIR}/{f}"):
        urllib.request.urlretrieve(f"{BASE}/{f}", f"{TRACE_DIR}/{f}")
print("downloaded:", sorted(os.listdir(TRACE_DIR)))

## Part 1 — Perfetto first (do NOT skip)

1. Open [ui.perfetto.dev](https://ui.perfetto.dev) → *Open trace file* → `vision_transformer/rank-0.json.gz` (Perfetto accepts the .gz directly).
2. Find the `ProfilerStep` annotations; measure one step's wall time (W/A/S/D to zoom/pan, M to mark a span).
3. Look at the GPU stream rows: which kernels dominate visually — compute (gemm/conv/elementwise) or `ncclKernel_*`?

**Commit these numbers before Part 2** (edit this cell):
- Step time (ms): `____`
- Fraction of the step where NO kernel is running on the GPU (eyeball): `____ %`
- Of kernel-busy time, roughly what fraction is NCCL vs compute? `____`
- Are NCCL kernels overlapped with compute, or mostly exposed? `____`
- Your one-sentence diagnosis of this job: `____`

In [ ]:
from hta.trace_analysis import TraceAnalysis
analyzer = TraceAnalysis(trace_dir=TRACE_DIR)

## Part 2 — Temporal breakdown: where does GPU time go?
Compute vs non-compute (communication + memory) vs idle, per rank. Check your idle guess against rank 0's row; check your NCCL-vs-compute guess against the compute/non-compute split. Then look *across* ranks: the idle spread (min to max rank) is your first straggler/imbalance signal.

In [ ]:
tb = analyzer.get_temporal_breakdown(visualize=False)
tb

## Part 3 — Communication–computation overlap
The single most interview-relevant number in a distributed trace: what fraction of communication time is hidden under compute. Healthy, well-bucketed DDP on a compute-dense model hides most of the all-reduce; the [pod-training answer](https://sys-design-primer-cvw.pages.dev/google-interview/6-answer-pod-training/) treats exposed-collective time as a first-class step-time segment. Compare the computed overlap % with your Part-1 exposed-vs-overlapped call, and ask: is low overlap here a *scheduling* failure (comm could hide but doesn't) or a *ratio* failure (there is simply more comm time than compute time to hide it under)? The Part-2 numbers answer that.

In [ ]:
overlap = analyzer.get_comm_comp_overlap(visualize=False)
overlap

## Part 4 — Idle-time attribution
Idle is not one thing: HTA splits it into **host_wait** (CPU didn't enqueue work — launch/input-bound) and **kernel_wait** (waiting on another kernel/stream — serialization). Returns a tuple; the first element is the breakdown.

In [ ]:
idle_df, idle_intervals = analyzer.get_idle_time_breakdown(ranks=[0], visualize=False)
idle_df

## Part 5 — Kernel-type breakdown: the verdict
`get_gpu_kernel_breakdown` returns two DataFrames: time by kernel *type* (communication / computation / memory, with overlap categories) and the top kernels per type. The type table is the one-glance verdict on this job.

In [ ]:
ktype_df, kernel_df = analyzer.get_gpu_kernel_breakdown(visualize=False, num_kernels=5)
print(ktype_df.to_string())
kernel_df.head(8)

## Part 6 — Write the narration
Five sentences, the [interview script](https://sys-design-primer-cvw.pages.dev/trace-reading/0-overview/): step time → the dominant time sink quantified → the signature named (which [fault-catalog row](https://sys-design-primer-cvw.pages.dev/trace-reading/tools/3-torch-profiler/)?) → the one confirming measurement → the fix and its expected win, with arithmetic. For the fix: if comm time exceeds compute time, better overlap scheduling alone cannot save you — what would? (Bigger per-GPU batch? Gradient compression? Fewer, larger buckets? Faster interconnect?) Pick one and defend it with the Part-2/Part-5 numbers.

## Stretch
- `analyzer.get_cuda_kernel_launch_stats()` — launch latency distribution: is the host also a suspect?
- `analyzer.get_memory_bw_summary()` — memcpy/memset bandwidth: is H2D input transfer implicated?
- `analyzer.get_queue_length_summary()` — stream queue depth: was the GPU ever starved of *enqueued* work (ties back to Part 4's host_wait)?
- Re-run Parts 2–5 on a trace you captured yourself (Lab C of [hands-on profiling](https://sys-design-primer-cvw.pages.dev/trace-reading/3-hands-on-profiling/)) and diff against this trace.

**What this artifact cannot teach** (know the boundary): MoE all-to-all patterns, recompilation storms, DVFS throttling, multi-*node* NCCL (this is single-node 8-GPU), and why any individual kernel is slow ([ncu's job](https://sys-design-primer-cvw.pages.dev/trace-reading/tools/2-ncu/)). API names drift between HTA versions; if a call errors, check `dir(analyzer)`.

---
## Answer key (ground truth — computed from these exact files)

<details><summary>Reveal only after Part 6</summary>

- **Temporal breakdown:** compute ≈ **29–30%** on every rank; non-compute ≈ **43–57%**; idle ≈ **13–27%** (rank 0 idles most at 27%, rank 3 least at 13.5%). Total kernel time ≈ 2.03 s per rank over the captured window.
- **Overlap:** comm-comp overlap ≈ **20–22%** on all ranks — poor, and consistent across ranks (so not a single straggler's fault).
- **Idle attribution (rank 0, compute stream):** **~71% host_wait, ~2% kernel_wait** — the CPU is not keeping the GPU fed on top of everything else.
- **Kernel types:** **COMMUNICATION 61.3%**, COMPUTATION 20.4%, computation-overlapping-communication 16.2%, MEMORY 2.1%.
- **The verdict:** a **communication-dominated job with structurally unhidable comm** — NCCL time (~61%) is ~3× pure compute time (~20%), so even perfect scheduling could hide at most a third of it. This is a *ratio* failure, not just a scheduling failure: the model is too small (per-GPU work too little) for its gradient volume at this interconnect speed. The catalog row is "unoverlapped gradient all-reduce," but the staff-level refinement is that the fix must change the ratio — bigger per-GPU batch (more compute per gradient byte), gradient compression/bucketing changes, or accepting that DDP on this model/hardware combination is interconnect-bound. Note the echo of the [scaling-book roofline](https://sys-design-primer-cvw.pages.dev/training/1-batch-size-primer/): per-device batch below the compute-bound floor — observed in the wild.

</details>